In [1]:
# Initialize Otter
import otter
grader = otter.Notebook("lab.ipynb")

c:\Users\IdeRy\miniforge3\envs\dsc80\lib\site-packages\otter\export\__init__.py:6: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


# Lab 8 – Feature Engineering ⚙️

## DSC 80, Fall 2026

### Due Date: Wednesday, November 25, 2026 at 11:59 p.m. Pacific

## Instructions

Welcome to the eighth DSC 80 lab this quarter!

Much like in DSC 10, this Jupyter Notebook contains the statements of the problems and provides code and Markdown cells to display your answers to the problems. Unlike DSC 10, the notebook is *only* for displaying a readable version of your final answers. The coding will be done in an accompanying `lab.py` file that is imported into the current notebook, and **you will only submit that `lab.py` file**, not this notebook!

Some additional guidelines:
- **Unlike in DSC 10, labs will have both public tests and hidden tests.** The bulk of your grade will come from your scores on hidden tests, which you will only see on Gradescope after the assignment deadline.
- **Do not change the function names in the `lab.py` file!** The functions in the `lab.py` file are how your assignment is graded, and they are graded by their name. If you changed something you weren't supposed to, you can find the original code in the [course GitHub repository](https://github.com/dsc-courses/dsc80-2026-fa).
- Notebooks are nice for testing and experimenting with different implementations before designing your function in your `lab.py` file. You can write code here, but make sure that all of your real work is in the `lab.py` file, since that's all you're submitting.
- You are encouraged to write your own additional helper functions to solve the lab, as long as they also end up in `lab.py`.

**To ensure that all of the work you want to submit is in `lab.py`, we've included a script named `lab-validation.py` in the lab folder. You shouldn't edit it, but instead, you should call it from the command line (e.g. the Terminal) to test your work.** More details on its usage are given at the bottom of this notebook.

**Importing code from `lab.py`**:

* Below, we import the `.py` file that's contained in the same directory as this notebook.
* We use the `autoreload` notebook extension to make changes to our `lab.py` file immediately available in our notebook. Without this extension, we would need to restart the notebook kernel to see any changes to `lab.py` in the notebook.
    - `autoreload` is necessary because, upon import, `lab.py` is compiled to bytecode (in the directory `__pycache__`). Subsequent imports of `lab` merely import the existing compiled python.

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
from lab import *

In [4]:
import pandas as pd
import numpy as np
np.set_printoptions(legacy='1.21')
import itertools
import plotly.express as px
import statsmodels.api as sm
from pathlib import Path
from sklearn.preprocessing import Binarizer, QuantileTransformer, FunctionTransformer

import warnings
warnings.filterwarnings('ignore')

## Part 1: Scaling Transformations 📐

A scaling transformation transforms the scale of the data of a particular quantitative column. Mathematically, each data point $d_i$ is replaced with a transformed value $t_i = f(d_i)$, where $f$ is a transformation function. We can transform any column in a dataset, whether it corresponds to a feature or a target.

Generally, the goal of a scaling transformation is to change the data from a complicated, non-linear relationship into a **linear** relationship. Linear relationships are very easy to understand and are easily used by models, like linear regression.

However, non-linear growth is a commonly seen relationship in data. Sometimes this growth is **polynomial** and sometimes it is **exponential**. The scaling transformations that turn these types of growth linear are **root** and **log** transformations respectively. (Generally, it is more difficult to determine which transformation is appropriate for a given dataset, though the [Tukey-Mosteller bulge diagram](https://freakonometrics.hypotheses.org/files/2014/06/Selection_005.png) is useful.)

Let's start by looking at some examples of transformations.

#### Example 1

Run the cell below to generate a scatter plot.

In [5]:
# By setting a seed, we guarantee that we will see the same results each time we run this cell.
np.random.seed(23)

# Generates a random scatter plot
x = np.arange(1, 101) + np.random.normal(0, 0.5, 100)
y = 2 * ((x + np.random.normal(0, 1, 100)) ** 2) + np.abs(x) * np.random.normal(0, 30, 100)
df_1 = pd.DataFrame().assign(x=x, y=y)

px.scatter(df_1, x='x', y='y', trendline="ols", trendline_color_override="red")

It doesn't appear to be the case that `'x'` and `'y'` are linearly associated here, and they aren't – there is a **quadratic** relationship between them. Note that if we were to create a **residual plot** above, there would be a pattern – the residuals for smaller `'x'` would mostly be positive, and the residuals for larger `'x'` would mostly be negative. From [DSC 10](https://inferentialthinking.com/chapters/15/5/Visual_Diagnostics.html), we know that patterns in a residual plot imply that the relationship between the two variables is non-linear.

To linearize the relationship, we can take the square root of each `'y'` value:

In [6]:
df_1['root y'] = np.sqrt(df_1['y'])

px.scatter(df_1, x='x', y='root y', trendline="ols", trendline_color_override="red")

That looks much better!

#### Example 2

Run the cell below to generate another scatter plot.

In [7]:
# By setting a seed, we guarantee that we will see the same results each time we run this cell
np.random.seed(32)

# Generates a different random scatter plot
x = np.linspace(2, 5, 100)
y = 10 * (np.e ** x) + np.abs(x) * np.random.normal(0, 5, 100) + np.random.normal(0, 30, 100)
df_2 = pd.DataFrame().assign(x=x, y=y)

px.scatter(df_2, x='x', y='y', trendline="ols", trendline_color_override="red")

Again, the relationship between `'x'` and `'y'` is not quite linear. Let's try the square root transformation we tried in Example 1:

In [8]:
df_2['root y'] = np.sqrt(df_2['y'])

px.scatter(df_2, x='x', y='root y', trendline="ols", trendline_color_override="red")

Hmm... the relationship certainly looks _more_ linear than before, but still not quite linear. Let's look at the residual plot:

In [9]:
# Feel free to use this function directly to help you answer Question 1.
def create_residual_plot(df, x, y):
    df = df.copy()
    from sklearn.linear_model import LinearRegression
    model = LinearRegression()
    model.fit(df[[x]], df[y])
    df['pred'] = model.predict(df[[x]])
    df[f'{y} residuals'] = df[y] - model.predict(df[[x]])
    return px.scatter(df, x='pred', y=f'{y} residuals', trendline='ols', trendline_color_override='red')

create_residual_plot(df_2, 'x', 'root y')

There is clearly a pattern in the residual plot. Let's instead try another transformation for the `'y'` values – $\log$.

In [10]:
df_2['log y'] = np.log(df_2['y'])

px.scatter(df_2, x='x', y='log y', trendline="ols", trendline_color_override="red")

That looks much better! We can verify that the residual plot has no "patterns":

In [11]:
create_residual_plot(df_2, 'x', 'log y')

Note – there is still evidence of **heteroscedasticity**, or "uneven spread", in this scatter plot, but the relationship is as close to linear as we'll get. If you look back at the cell in which we generated the data for this scatter plot, you’ll see that it was created using the exponential function `np.e ** x`. In real life, we won’t get to see how our data was created, but we’ll still need to figure out how to linearize it!

### Question 1 –  Root vs. Log

Now that we've learned how to perform transformations with example datasets, it's your job to apply these ideas to a real dataset. Below, you are given a dataset that describes the [number of home runs in Major League Baseball per year](https://www.baseball-almanac.com/hitting/hihr6.shtml). The relationship between the two variables, `'Year'` and `'Home Runs'`, is not linear.

Specifically, your job is to determine the **best transformation to apply to the `'Home Runs'` column**, in order to linearize the relationship as much as possible.

Complete the implementation of the function `best_transformation`, which returns either 1, 2, 3, or 4, with the value corresponding to one of the following choices for transforming `'Home Runs'`:

1. Square root transformation ($x \rightarrow \sqrt x$)
2. Reciprocal transformation ($x \rightarrow \frac1x$)
3. Log transformation ($x \rightarrow \log x$)
4. Square transformation ($x \rightarrow x^2$)

In [12]:
homeruns_fp = Path('data')/'homeruns.csv'
homeruns = pd.read_csv(homeruns_fp)
homeruns

,Year,Home Runs
0,1900,254
1,1901,455
2,1902,354
3,1903,335
4,1904,331
...,...,...
121,2021,5940
122,2022,5215
123,2023,5868
124,2024,5453


In [13]:
px.scatter(homeruns, x='Year', y='Home Runs', trendline="ols", trendline_color_override="red")


In [14]:
homeruns['Transformed'] = np.sqrt(homeruns['Home Runs'])

px.scatter(homeruns, x='Year', y='Transformed', trendline="ols", trendline_color_override="red")

In [15]:
create_residual_plot(homeruns, 'Year', 'Transformed')

In [16]:
def best_transformation():
    return 1


In [17]:
grader.check("q1")

q1 results: All test cases passed!

## Part 2: Diamond Pricing 💎

In this next section, you will pretend you are a jewelry appraiser and predict the prices of diamonds given several standard characteristics of diamonds.

You will use linear regression to predict prices, while improving the quality of your predictions using **feature engineering**. Since this question is supposed to help you understand feature engineering, **you will be building these features from scratch, instead of using the built in `sklearn` or `pandas` methods**.

The `diamonds` dataset is loaded in for you below. The DataFrame has 53940 rows and 10 columns:

|column|description|unique values or range|
|---|---|---|
|`'carat'`|weight of the diamond in carats (each carat is 0.2 grams)| 0.2 - 5.01 |
|`'cut'`|quality of the cut | Fair, Good, Very Good, Premium, Ideal |
|`'color'`|diamond color | J (worst), I, H, G, F, E, D (best) |
|`'clarity'`|a measurement of how clear the diamond is | I1 (worst), SI2, SI1, VS2, VS1, VVS2, VVS1, IF (best) |
|`'depth'`|total depth percentage, computed as z / mean(x, y) = 2 * z / (x + y) | 43 - 79 |
|`'table'`|width of top of diamond relative to widest point | 43 - 95 |
|`'price'`|price in US dollars | \\$326 - \\$18,823 USD |
|`'x'`|length in mm | 0 - 10.74 |
|`'y'`|width in mm | 0 - 58.9 | 
|`'z'`|depth in mm | 0 - 31.8 |

If you want to learn more about how diamonds are measured, refer to [this page by the American Gem Society](https://www.americangemsociety.org/4cs-of-diamonds/).

In [18]:
diamonds = pd.read_csv(Path('data')/'diamonds.csv')
diamonds.head()

,carat,cut,color,clarity,depth,table,price,x,y,z
0,0.23,Ideal,E,SI2,61.5,55.0,326,3.95,3.98,2.43
1,0.21,Premium,E,SI1,59.8,61.0,326,3.89,3.84,2.31
2,0.23,Good,E,VS1,56.9,65.0,327,4.05,4.07,2.31
3,0.29,Premium,I,VS2,62.4,58.0,334,4.20,4.23,2.63
4,0.31,Good,J,SI2,63.3,58.0,335,4.34,4.35,2.75


### Question 2 – Ordinal Encoding 🔢

Every categorical variable in the dataset is an ordinal column, meaning that there is an inherent order that we can use to sort the values in the column. Recall that **ordinal encoding** is a feature transformation that maps the values in an ordinal column to integers in a way that preserves the order of the column values. For instance, an ordinal encoding for Freshman, Sophomore, Junior, Senior is 0, 1, 2, 3.

Complete the implementation of the function `create_ordinal`, which takes in the `diamonds` DataFrame and returns a DataFrame of ordinal features only with names of the form `'ordinal_<col>'`, where `'<col>'` is the original categorical column name. For instance, the `'ordinal_color'` column should consist of values from 0 to 6, where 0 refers to `'J'` and 6 refers to `'D'`. (In all cases, start counting from 0.)

***Notes:*** 
- Remember, you must create this function using basic `pandas`, without using `sklearn`.
- Don't include non-ordinal features in the returned DataFrame. That is, if there are only three columns in `diamonds` that are ordinal, `create_ordinal` should return a DataFrame with three columns.
- You will need to refer to the data dictionary above. For each ordinal column, the unique values are listed in order.

In [19]:
def create_ordinal(df):

    map_list = [
        {
        'Fair' : 0,
        'Good' : 1,
        'Very Good' : 2,
        'Premium' : 3,
        'Ideal' : 4
        },
        {
            'J' : 0,
            'I' : 1,
            'H' : 2,
            'G' : 3,
            'F' : 4,
            'E' : 5,
            'D' : 6
        },
        {
            'I1' : 0,
            'SI2' : 1,
            'SI1' : 2,
            'VS2' : 3,
            'VS1' : 4,
            'VVS2' : 5,
            'VVS1' : 6,
            'IF' : 7
        }
    ]

    new = pd.DataFrame()

    for i, column in enumerate(['cut', 'color', 'clarity']):
        new[f'ordinal_{column}'] = df[column].map(map_list[i])

    return new

In [20]:
# don't change this cell, but do run it -- it is needed for the tests to work
diamonds = pd.read_csv(Path('data')/'diamonds.csv')
out_q2 = create_ordinal(diamonds)
out_q2

,ordinal_cut,ordinal_color,ordinal_clarity
0,4,5,1
1,3,5,2
2,1,5,4
3,3,1,3
4,1,0,1
...,...,...,...
53935,4,6,2
53936,1,6,2
53937,2,6,2
53938,3,2,1


In [21]:
grader.check("q2")

q2 results: All test cases passed!

### Question 3 – Nominal Encoding 📊

Even though the categorical variables in the dataset are ordinal, we can still treat them as nominal by forgetting their order. To treat the categorical variables in our dataset as nominal, we might **one-hot encode** them. 

#### `create_one_hot`

Complete the implementation of the function `create_one_hot`, which takes in the `diamonds` DataFrame and returns a DataFrame of one-hot encoded features with names of the form `'one_hot_<col>_<val>'`, where `'<col>'` is the original categorical column name, and `'<val>'` is the value found in the categorical column `'<col>'`. For instance, one of your column names will be `'one_hot_color_J'`.

***Notes***:
- Only include one-hot-encoded columns in the output DataFrame.
- **Do not** use `sklearn` or `pd.get_dummies` for this question!
- As per usual, write an efficient implementation. You may use a `for`-loop over **columns** and **unique values**, but not over rows. And the order of columns does not matter.
- In lecture, we will look at cases where we need to drop one one-hot encoded column per categorical variable. **Do not drop** any one-hot encoded columns here!

<br>

#### `create_proportions`

Similar to the one-hot encoding case, you can replace a value in a nominal column with the proportion of times that value appears in the column. For instance, if a column consists of the values `['a', 'b', 'a', 'c']`, then the proportion-encoded column is `[0.5, 0.25, 0.5, 0.25]`.  This might be a reasonable approach to predicting the price of a diamond, as you might expect *rarer attributes to be considered more valuable* than common ones.

Complete the implementation of the function `create_proportions`, which takes in the `diamonds` DataFrame and returns a DataFrame of proportion-encoded features with names of the form `'proportion_<col>'`, where `'<col>'` is the original categorical column name.

In [22]:
def create_one_hot(df):
    new = pd.DataFrame()

    for column in ['cut', 'color', 'clarity']:
        for i, subcolumn in enumerate(df[column].unique()):
            new[f'one_hot_{column}_{subcolumn}'] = (df[column] == subcolumn).astype(int)

    return new

def create_proportions(df):
    new = pd.DataFrame()
    ordinals = create_ordinal(df)

    for column in ['cut', 'color', 'clarity']:
            new[f'proportion_{column}'] = ordinals[f'ordinal_{column}'].map(ordinals[f'ordinal_{column}'].value_counts(normalize=True))

    return new


In [23]:
# don't change this cell, but do run it -- it is needed for the tests to work
diamonds = pd.read_csv(Path('data')/'diamonds.csv')
out1_q3 = create_one_hot(diamonds)
out2_q3 = create_proportions(diamonds)

In [24]:
out_q2['ordinal_cut'].unique()

array([4, 3, 1, 2, 0])

In [25]:
out1_q3

,one_hot_cut_Ideal,one_hot_cut_Premium,one_hot_cut_Good,one_hot_cut_Very Good,one_hot_cut_Fair,one_hot_color_E,one_hot_color_I,one_hot_color_J,one_hot_color_H,one_hot_color_F,one_hot_color_G,one_hot_color_D,one_hot_clarity_SI2,one_hot_clarity_SI1,one_hot_clarity_VS1,one_hot_clarity_VS2,one_hot_clarity_VVS2,one_hot_clarity_VVS1,one_hot_clarity_I1,one_hot_clarity_IF
0,1,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0
1,0,1,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0
2,0,0,1,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0
3,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0
4,0,0,1,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
53935,1,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0
53936,0,0,1,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0
53937,0,0,0,1,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0
53938,0,1,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0


In [26]:
out_q2['ordinal_cut'].value_counts(normalize=True)

ordinal_cut
4    0.399537
3    0.255673
2    0.223990
1    0.090953
0    0.029848
Name: proportion, dtype: float64

In [27]:
out2_q3

,proportion_cut,proportion_color,proportion_clarity
0,0.399537,0.181628,0.170449
1,0.255673,0.181628,0.242214
2,0.090953,0.181628,0.151483
3,0.255673,0.100519,0.227253
4,0.090953,0.052058,0.170449
...,...,...,...
53935,0.399537,0.125603,0.242214
53936,0.090953,0.125603,0.242214
53937,0.223990,0.125603,0.242214
53938,0.255673,0.153949,0.170449


In [28]:
grader.check("q3")

q3 results: All test cases passed!

### Question 4 – Quadratic Features 📈

Linear regression doesn't capture non-linear relationships between variables. However, you can create features that encode such dependencies **before** fitting your regression model. Creating polynomial features is one way to do this. For example, the diamonds dataset contains `'x'`, `'y'`, and `'z'` dimensions for each stone. However, different combinations of size may be more valuable than others: a "deep and wide" diamond might be considered more valuable than a shallow, but "long and wide" diamond.

Complete the implementation of the function `create_quadratics`, which takes in the `diamonds` DataFrame and returns a DataFrame of quadratic features of the form `'<col1> * <col2>'`, where `'<col1>'` and `'<col2>'` are the original quantitative columns. The output DataFrame should contain a column for every distinct pair of quantitative columns in `diamonds` (aside from `price`, which should be left out as it is what we are predicting). For instance, one of the columns in the returned DataFrame should be named either `'carat * x'` or `'x * carat'`; the order of column names is not important.

***Notes***:
- Again, **do not** use `sklearn` for this question! 
- Find all pairs of quantitative columns efficiently, without using a nested loop (hint: we've already imported `itertools` for you). Our solution contains just a single `for`-loop, over pairs of columns.
- The columns of the resulting DataFrame may be in any order.

In [29]:
def create_quadratics(df):
    quantitative_variables = ['carat', 'depth', 'table', 'x', 'y', 'z']

    return pd.DataFrame(
        {
            f'{combination[0]} * {combination[1]}' : 
            df[combination[0]].astype(float)*df[combination[1]].astype(float)
            for combination in itertools.combinations(quantitative_variables, 2)
        }
    )

In [30]:
[combination for combination in itertools.combinations(['meow', 'bark', 'moo'], 2)]

[('meow', 'bark'), ('meow', 'moo'), ('bark', 'moo')]

In [31]:
# don't change this cell, but do run it -- it is needed for the tests to work
diamonds = pd.read_csv(Path('data')/'diamonds.csv')
out_q4 = create_quadratics(diamonds)

In [32]:
tupl = ('carat', 'depth')
diamonds[tupl[0]].astype(float)*diamonds[tupl[1]].astype(float)

0        14.145
1        12.558
2        13.087
3        18.096
4        19.623
          ...  
53935    43.776
53936    45.432
53937    43.960
53938    52.460
53939    46.650
Length: 53940, dtype: float64

In [33]:
out_q4

,carat * depth,carat * table,carat * x,carat * y,carat * z,depth * table,depth * x,depth * y,depth * z,table * x,table * y,table * z,x * y,x * z,y * z
0,14.145,12.65,0.9085,0.9154,0.5589,3382.5,242.925,244.770,149.445,217.25,218.90,133.65,15.7210,9.5985,9.6714
1,12.558,12.81,0.8169,0.8064,0.4851,3647.8,232.622,229.632,138.138,237.29,234.24,140.91,14.9376,8.9859,8.8704
2,13.087,14.95,0.9315,0.9361,0.5313,3698.5,230.445,231.583,131.439,263.25,264.55,150.15,16.4835,9.3555,9.4017
3,18.096,16.82,1.2180,1.2267,0.7627,3619.2,262.080,263.952,164.112,243.60,245.34,152.54,17.7660,11.0460,11.1249
4,19.623,17.98,1.3454,1.3485,0.8525,3671.4,274.722,275.355,174.075,251.72,252.30,159.50,18.8790,11.9350,11.9625
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
53935,43.776,41.04,4.1400,4.1472,2.5200,3465.6,349.600,350.208,212.800,327.75,328.32,199.50,33.1200,20.1250,20.1600
53936,45.432,39.60,4.0968,4.1400,2.5992,3470.5,359.039,362.825,227.791,312.95,316.25,198.55,32.7175,20.5409,20.7575
53937,43.960,42.00,3.9620,3.9760,2.4920,3768.0,355.448,356.704,223.568,339.60,340.80,213.60,32.1488,20.1496,20.2208
53938,52.460,49.88,5.2890,5.2632,3.2164,3538.0,375.150,373.320,228.140,356.70,354.96,216.92,37.6380,23.0010,22.8888


In [34]:
grader.check("q4")

q4 results: All test cases passed!

### Question 5 – Comparing Performance 🏆

We've now created several sets of features. **Which features are best able to predict the price of a diamond in a linear regression model?** We'll look at both single-feature linear regression models and multiple regression models. In all cases, use the default arguments to `sklearn`'s `LinearRegression` object (i.e. assume there is an intercept term).

<br>

Two of the evaluation metrics we can use to compare regression models are RMSE and $R^2$.

RMSE, or Root Mean Squared Error, roughly measures the average distance between a model's predicted values and the actual values. The RMSE indicates how concentrated predictions are around the line of best fit, with lower values typically indicating smaller residuals and better model performance. RMSE can be calculated using the formula below, where $n$ is the number of data points, $\hat{y_i}$ is the $i$-th prediction, and $y_i$ is the corresponding actual value.

$$\text{RMSE} = \sqrt{\frac{\displaystyle\sum_{i=1}^n (y_i - \hat{y_i})^2}{n}}$$

Remember that in least squares regression, we choose the model parameters that minimize RMSE.

<br>

$R^2$, also known as the coefficient of determination, denotes the proportion of variance in the dependent variable that the independent variable explains. It is used to assess the quality of a linear fit, and in the simple linear regression case, it is the square of the correlation coefficient $r$. $R^2$ values range between 0 and 1, with lower values indicating a worse linear fit and with higher values indicating a better linear fit. If `model` is a fitted `sklearn` regression object, $R^2$ can be computed by calling `model.score`. 

#### Single-Feature Models

- (1) Fit a single-feature linear regression model on `'carat'`. What is the $R^2$ of the model? (Note that `'carat'` turns out to be the best single feature to use in a linear model that predicts price.)
- (2) What is the RMSE of the model you created in (1)?
- (3) Among the other **quantitative** features present in the original `diamonds`, which produces the single-feature linear regression model with the highest $R^2$?
- (4) Among all the new features you created in Questions 2-4, which produces the single-feature linear regression model with the highest $R^2$?
- (5) Among the new categorical features you created in Question 2 and 3, which produces the single-feature linear regression model with the highest $R^2$? 

#### Multiple Regression

Now, fit a multiple regression model using:
- the quantitative columns that were present in the original `diamonds` dataset, and
- the quantitative features engineered in Question 4

as features. (Don't use any of the encodings of categorical columns from Questions 2 and 3.)

- (6) What is the RMSE of this new model?


<br>

Complete the implementation of the function `comparing_performance`, which returns a **hard-coded list** containing the answers to the six questions numbered (1), (2), ..., (6) above. Numerical values should not be rounded. String values should be column names.

***Hint:*** Repeatedly use the `sklearn` pattern included below. It's a good idea to make a helper function that takes in a column, performs single-feature regression using the input column as the feature, and returns the $R^2$ and RMSE of the model.

In [35]:
from sklearn.linear_model import LinearRegression

def fit_and_describe(x, y, df_x, df_y=diamonds):

    X = df_x[[x]]
    y = df_y[y]

    lr = LinearRegression()
    lr.fit(X, y)  # X is a DataFrame of training data; y is a Series of prices

    print(f'R-squared:\n{lr.score(X, y)}\n') # R-squared
    # print(f'RMSE:\n{np.sqrt(np.sum((lr.predict(X) - y.to_numpy())**2)/len(y))}\n') # RMSE
    # print(f'Predicted:\n{lr.predict(X)}\n') # predicted prices

fit_and_describe('carat', 'price', diamonds)

R-squared:
0.8493305264354858



In [36]:
diamonds

,carat,cut,color,clarity,depth,table,price,x,y,z
0,0.23,Ideal,E,SI2,61.5,55.0,326,3.95,3.98,2.43
1,0.21,Premium,E,SI1,59.8,61.0,326,3.89,3.84,2.31
2,0.23,Good,E,VS1,56.9,65.0,327,4.05,4.07,2.31
3,0.29,Premium,I,VS2,62.4,58.0,334,4.20,4.23,2.63
4,0.31,Good,J,SI2,63.3,58.0,335,4.34,4.35,2.75
...,...,...,...,...,...,...,...,...,...,...
53935,0.72,Ideal,D,SI1,60.8,57.0,2757,5.75,5.76,3.50
53936,0.72,Good,D,SI1,63.1,55.0,2757,5.69,5.75,3.61
53937,0.70,Very Good,D,SI1,62.8,60.0,2757,5.66,5.68,3.56
53938,0.86,Premium,H,SI2,61.0,58.0,2757,6.15,6.12,3.74


In [37]:
for column in ['carat', 'depth', 'table', 'x', 'y', 'z']:
    fit_and_describe(column, 'price', diamonds)

R-squared:
0.8493305264354858

R-squared:
0.00011336722437849112

R-squared:
0.016163029068700707

R-squared:
0.782225554041623

R-squared:
0.7489533304600553

R-squared:
0.7417506045344293



In [38]:
for column in out_q2.columns:
    fit_and_describe(column, 'price', out_q2, diamonds)

R-squared:
0.002861250800106463

R-squared:
0.029760020332370685

R-squared:
0.021550260868784754



In [39]:
for column in out1_q3.columns:
    fit_and_describe(column, 'price', out1_q3, diamonds)

R-squared:
0.00944305542308832

R-squared:
0.009159633114764865

R-squared:
9.735619044271004e-08

R-squared:
4.347408039628853e-05

R-squared:
0.0003507462354399582

R-squared:
0.010219060389604184

R-squared:
0.009433310013271745

R-squared:
0.006676582841153356

R-squared:
0.0035073480221267106

R-squared:
0.0005837473010588123

R-squared:
7.320729004378634e-05

R-squared:
0.005252269649564267

R-squared:
0.016491771831517066

R-squared:
8.02212894741583e-05

R-squared:
9.773810483015843e-05

R-squared:
1.1271812626612032e-06

R-squared:
0.0027437514855134237

R-squared:
0.00907564226491464

R-squared:
6.519650574965397e-08

R-squared:
0.002459763910689916



In [40]:
for column in out2_q3.columns:
    fit_and_describe(column, 'price', out2_q3, diamonds)

R-squared:
0.005004173762515052

R-squared:
0.008312773999915812

R-squared:
0.006333681179798156



In [41]:
for column in out_q4.columns:
    fit_and_describe(column, 'price', out_q4, diamonds)

R-squared:
0.8424152904994928

R-squared:
0.8391935530253825

R-squared:
0.852427068053341

R-squared:
0.8440006962019663

R-squared:
0.8463424574270608

R-squared:
0.014442089716740991

R-squared:
0.7720372927441929

R-squared:
0.7427546685902617

R-squared:
0.7105098503030792

R-squared:
0.7399855739149872

R-squared:
0.7145971865557245

R-squared:
0.717872481105092

R-squared:
0.8120750219043602

R-squared:
0.8163133975201253

R-squared:
0.7741499677942056



In [42]:
X = pd.concat([diamonds[['carat', 'depth', 'table', 'x', 'y', 'z']], out_q4], axis=1)
y = diamonds['price']

lr = LinearRegression()
lr.fit(X, y)  # X is a DataFrame of training data; y is a Series of prices

print(f'RMSE:\n{np.sqrt(np.sum((lr.predict(X) - y.to_numpy())**2)/len(y))}\n') # RMSE

RMSE:
1434.8400089047336



In [43]:
X

,carat,depth,table,x,y,z,carat * depth,carat * table,carat * x,carat * y,...,depth * table,depth * x,depth * y,depth * z,table * x,table * y,table * z,x * y,x * z,y * z
0,0.23,61.5,55.0,3.95,3.98,2.43,14.145,12.65,0.9085,0.9154,...,3382.5,242.925,244.770,149.445,217.25,218.90,133.65,15.7210,9.5985,9.6714
1,0.21,59.8,61.0,3.89,3.84,2.31,12.558,12.81,0.8169,0.8064,...,3647.8,232.622,229.632,138.138,237.29,234.24,140.91,14.9376,8.9859,8.8704
2,0.23,56.9,65.0,4.05,4.07,2.31,13.087,14.95,0.9315,0.9361,...,3698.5,230.445,231.583,131.439,263.25,264.55,150.15,16.4835,9.3555,9.4017
3,0.29,62.4,58.0,4.20,4.23,2.63,18.096,16.82,1.2180,1.2267,...,3619.2,262.080,263.952,164.112,243.60,245.34,152.54,17.7660,11.0460,11.1249
4,0.31,63.3,58.0,4.34,4.35,2.75,19.623,17.98,1.3454,1.3485,...,3671.4,274.722,275.355,174.075,251.72,252.30,159.50,18.8790,11.9350,11.9625
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
53935,0.72,60.8,57.0,5.75,5.76,3.50,43.776,41.04,4.1400,4.1472,...,3465.6,349.600,350.208,212.800,327.75,328.32,199.50,33.1200,20.1250,20.1600
53936,0.72,63.1,55.0,5.69,5.75,3.61,45.432,39.60,4.0968,4.1400,...,3470.5,359.039,362.825,227.791,312.95,316.25,198.55,32.7175,20.5409,20.7575
53937,0.70,62.8,60.0,5.66,5.68,3.56,43.960,42.00,3.9620,3.9760,...,3768.0,355.448,356.704,223.568,339.60,340.80,213.60,32.1488,20.1496,20.2208
53938,0.86,61.0,58.0,6.15,6.12,3.74,52.460,49.88,5.2890,5.2632,...,3538.0,375.150,373.320,228.140,356.70,354.96,216.92,37.6380,23.0010,22.8888


In [ ]:
def comparing_performance():
    return [
        0.8493305264354858, 
        1548.5331930613174,
        'x',
        'carat * x',
        'ordinal_color',
        1434.8400089047336
    ]

In [61]:
# don't change this cell, but do run it -- it is needed for the tests to work
import numbers
out_q5 = comparing_performance()
out_q5

[0.8493305264354858,
 1548.5331930613174,
 'carat',
 'carat * x',
 'ordinal_color',
 1434.8400089047336]

In [62]:
grader.check("q5")

q5 results: All test cases passed!

## Part 3 – Feature Engineering with `sklearn` 🧠

In this final question, you will use `sklearn`'s transformers and estimators for feature engineering. While everything you do with `sklearn` is possible to do with `pandas`, `sklearn` transformers enable you to couple your feature engineering with your modeling. This will allow you to more quickly build and assess your models in `sklearn`.

Specifically, you will create a `TransformDiamonds` class that contains the three methods specified below – `transform_carat` (6.1), `transform_to_quantile` (6.2), and `transform_to_depth_pct` (6.3). In the starter code, there is a skeleton for `TransformDiamonds` that is initialized with a DataFrame `diamonds`.

Each of the methods you implement in the `TransformDiamonds` class should take in a DataFrame, initialize a specific `sklearn.Transformer` object (like `Binarizer` or `FunctionTransformer`), and use the transformer to transform columns from the input DataFrame. You should **not** use DataFrame methods like `apply` in this problem.

In [46]:
from sklearn.preprocessing import Binarizer, QuantileTransformer, FunctionTransformer

Question 6 is made up of the three subparts below.

### Question 6.1 – Transforming a Quantitative Column into a Binary Column (`transform_carat`)

We call a diamond **large** if its weight is strictly greater than 1 carat. We want to **binarize** weights, so that they are 1 for large diamonds and 0 for small diamonds. Complete the implementation of the method `transform_carat`, which takes in a DataFrame like `diamonds` and returns a binarized **array** of weights. Use a `Binarizer` object as your transformer.

***Notes***:
- You will return an array, not a Series, because `sklearn` thinks in terms of `np.ndarray`s, not DataFrames.
- The implementation of this function should only take two lines.

In [47]:
# don't change this cell, but do run it -- it is needed for the tests to work
diamonds = pd.read_csv(Path('data')/'diamonds.csv')
q6a_trans = TransformDiamonds(diamonds)
q6a_out = q6a_trans.transform_carat(diamonds)

In [48]:
grader.check("q6.1")

q6.1 results:
    q6.1 - 1 result:
        Trying:
            isinstance(q6a_out, np.ndarray)
        Expecting:
            True
        **********************************************************************
        Line 1, in q6.1 0
        Failed example:
            isinstance(q6a_out, np.ndarray)
        Expected:
            True
        Got:
            False

    q6.1 - 2 result:
        Trying:
            bool(q6a_out[172, 0] == 1)
        Expecting:
            True
        **********************************************************************
        Line 1, in q6.1 1
        Failed example:
            bool(q6a_out[172, 0] == 1)
        Exception raised:
            Traceback (most recent call last):
              File "c:\Users\IdeRy\miniforge3\envs\dsc80\lib\doctest.py", line 1350, in __run
                exec(compile(example.source, filename, "single",
              File "<doctest q6.1 1[0]>", line 1, in <module>
                bool(q6a_out[172, 0] == 1)
            TypeError: 'NoneType' object is not subscriptable

    q6.1 - 3 result:
        Trying:
            bool(q6a_out[0, 0] == 0)
        Expecting:
            True
        **********************************************************************
        Line 1, in q6.1 2
        Failed example:
            bool(q6a_out[0, 0] == 0)
        Exception raised:
            Traceback (most recent call last):
              File "c:\Users\IdeRy\miniforge3\envs\dsc80\lib\doctest.py", line 1350, in __run
                exec(compile(example.source, filename, "single",
              File "<doctest q6.1 2[0]>", line 1, in <module>
                bool(q6a_out[0, 0] == 0)
            TypeError: 'NoneType' object is not subscriptable

### Question 6.2 – Transforming a Quantitative Column into Quantiles (`transform_to_quantile`)

You will now transform the `'carat'` column so that each diamond's weight in carats is replaced with the **percentile** in which its weight lies.

Complete the implementation of the method `transform_to_quantiles`, which takes in a DataFrame like `diamonds` and returns an array containing the percentiles of the weight of each diamond, among all diamonds in `self.data`. This array should consist of proportions between 0 and 1; for instance, 0.65 will refer to the 65th percentile. The relevant transformer is `QuantileTransformer`. 

Some guidance:

- Unlike with `Binarizer`, you need to `fit` your `QuantileTransformer` before calling `transform` on the input DataFrame `data`. 
    - You should `fit` your transformer on the DataFrame `self.data`, but you should only `transform` the `data` that is passed to `transform_to_quantiles`. 
    - Note that these two DataFrames, `self.data` and `data`, don't have to be the same! For instance, in the last two lines of the testing setup cell below, we fit a `QuantileTransformer` using just the first 1000 rows of `diamonds`, and then `transform` the entire `diamonds` DataFrame. Make sure your `transform_to_quantiles` method works in such a case.
- When initializing your `QuantileTransformer`, use `n_quantiles=100`.

In [49]:
# don't change this cell, but do run it -- it is needed for the tests to work
q6b_trans = TransformDiamonds(diamonds)
q6b_out = q6b_trans.transform_to_quantile(diamonds)
q6b_trans_top_1000 = TransformDiamonds(diamonds[:1000])
q6b_out_top_1000 = q6b_trans_top_1000.transform_to_quantile(diamonds)

In [50]:
grader.check("q6.2")

q6.2 results:
    q6.2 - 1 result:
        Trying:
            isinstance(q6b_out, np.ndarray)
        Expecting:
            True
        **********************************************************************
        Line 1, in q6.2 0
        Failed example:
            isinstance(q6b_out, np.ndarray)
        Expected:
            True
        Got:
            False

    q6.2 - 2 result:
        Trying:
            bool(np.isclose(q6b_out[0, 0], 0.0075757, atol=0.0001))
        Expecting:
            True
        **********************************************************************
        Line 1, in q6.2 1
        Failed example:
            bool(np.isclose(q6b_out[0, 0], 0.0075757, atol=0.0001))
        Exception raised:
            Traceback (most recent call last):
              File "c:\Users\IdeRy\miniforge3\envs\dsc80\lib\doctest.py", line 1350, in __run
                exec(compile(example.source, filename, "single",
              File "<doctest q6.2 1[0]>", line 1, in <module>
                bool(np.isclose(q6b_out[0, 0], 0.0075757, atol=0.0001))
            TypeError: 'NoneType' object is not subscriptable

    q6.2 - 3 result:
        Trying:
            bool(np.isclose(q6b_out[1, 0], 0.0025252, atol=0.0001))
        Expecting:
            True
        **********************************************************************
        Line 1, in q6.2 2
        Failed example:
            bool(np.isclose(q6b_out[1, 0], 0.0025252, atol=0.0001))
        Exception raised:
            Traceback (most recent call last):
              File "c:\Users\IdeRy\miniforge3\envs\dsc80\lib\doctest.py", line 1350, in __run
                exec(compile(example.source, filename, "single",
              File "<doctest q6.2 2[0]>", line 1, in <module>
                bool(np.isclose(q6b_out[1, 0], 0.0025252, atol=0.0001))
            TypeError: 'NoneType' object is not subscriptable

### Question 6.3 – Transforming a Quantitative Column Using a Formula

Recall from the introduction to Part 2 that the "depth percentage" of a diamond is defined as

$$\text{Depth Pct.} = 100\% \cdot \frac{2z}{x + y}$$

where $x$, $y$, and $z$ come from the `'x'`, `'y'`, and `'z'` columns in `diamonds`.

Let's suppose that for some reason we don't have access to the `'depth'` column in `diamonds`, and instead need to recreate it just by looking at the `'x'`, `'y'`, and `'z'` columns. 

Complete the implementation of the method `transform_to_depth_pct`, which takes in a DataFrame like `diamonds` and returns an array consisting of the depth percentages of each diamond. Percentages should be between 0 and 100. The relevant transformer is `FunctionTransformer`.

***Notes***:
- To use `FunctionTransformer`, you will need to define your own function that takes in a 2D array and returns a single array.
- Don't worry about missing values or dividing by zero. You can ignore the possibility of such issues for this question.
- To verify your work, compare your outputted array to the actual `'depth'` column in `diamonds`. The results will not be exactly the same, due to rounding, but should be similar.
- It may seem like `FunctionTransformer` is totally unnecessary, since we can compute depth percentages using broadcasting directly. However, as we will see in lecture, transformers can be **pipelined** with other processing steps which can greatly simplify our code.

In [51]:
# don't change this cell, but do run it -- it is needed for the tests to work
diamonds = pd.read_csv(Path('data')/'diamonds.csv')
q6c_trans = TransformDiamonds(diamonds)
q6c_out = q6c_trans.transform_to_depth_pct(diamonds)

In [52]:
grader.check("q6.3")

q6.3 results:
    q6.3 - 1 result:
        Trying:
            len(q6c_out.shape) == 1
        Expecting:
            True
        **********************************************************************
        Line 1, in q6.3 0
        Failed example:
            len(q6c_out.shape) == 1
        Exception raised:
            Traceback (most recent call last):
              File "c:\Users\IdeRy\miniforge3\envs\dsc80\lib\doctest.py", line 1350, in __run
                exec(compile(example.source, filename, "single",
              File "<doctest q6.3 0[0]>", line 1, in <module>
                len(q6c_out.shape) == 1
            AttributeError: 'NoneType' object has no attribute 'shape'

    q6.3 - 2 result:
        Trying:
            bool(np.isclose(q6c_out[0], 61.286, atol=0.0001))
        Expecting:
            True
        **********************************************************************
        Line 1, in q6.3 1
        Failed example:
            bool(np.isclose(q6c_out[0], 61.286, atol=0.0001))
        Exception raised:
            Traceback (most recent call last):
              File "c:\Users\IdeRy\miniforge3\envs\dsc80\lib\doctest.py", line 1350, in __run
                exec(compile(example.source, filename, "single",
              File "<doctest q6.3 1[0]>", line 1, in <module>
                bool(np.isclose(q6c_out[0], 61.286, atol=0.0001))
            TypeError: 'NoneType' object is not subscriptable

## Congratulations! You're done with Lab 8! 🏁

As a reminder, all of the work you want to submit needs to be in `lab.py`.

To ensure that all of the work you want to submit is in `lab.py`, we've included a script named `lab-validation.py` in the lab folder. You shouldn't edit it, but instead, you should call it from the command line (e.g. the Terminal) to test your work.

Once you've finished the lab, you should open the command line and run, in the directory for this lab:

```
python lab-validation.py
```

**This will run all of the `grader.check` cells that you see in this notebook, but only using the code in `lab.py` – that is, it doesn't look at any of the code in this notebook. If all of your `grader.check` cells pass in this notebook but not all of them pass in your command line with the above command, then you likely have code in your notebook that isn't in your `lab.py`!**

You can also use `lab-validation.py` to test individual questions. For instance,

```
python lab-validation.py q1 q2 q4
```

will run the `grader.check` cells for Questions 1, 2, and 4 – again, only using the code in `lab.py`. [This video](https://www.loom.com/share/0ea254b85b2745e59322b5e5a8692e91?sid=5acc92e6-0dfe-4555-9b6a-8115b6a52f99) how to use the script as well.

Once `python lab-validation.py` shows that you're passing all test cases, you're ready to submit your `lab.py` (and only your `lab.py`) to Gradescope. Once submitting to Gradescope, make sure to stick around until all test cases pass.

There is also a call to `grader.check_all()` below in _this_ notebook, but make sure to also follow the steps above.

---

To double-check your work, the cell below will rerun all of the autograder tests.

In [53]:
grader.check_all()

q1 results: All test cases passed!

q2 results: All test cases passed!

q3 results: All test cases passed!

q4 results: All test cases passed!

q5 results:
    q5 - 1 result:
        Trying:
            len(out_q5) == 6
        Expecting:
            True
        **********************************************************************
        Line 1, in q5 0
        Failed example:
            len(out_q5) == 6
        Exception raised:
            Traceback (most recent call last):
              File "c:\Users\IdeRy\miniforge3\envs\dsc80\lib\doctest.py", line 1350, in __run
                exec(compile(example.source, filename, "single",
              File "<doctest q5 0[0]>", line 1, in <module>
                len(out_q5) == 6
            TypeError: object of type 'NoneType' has no len()

    q5 - 2 result:
        Trying:
            isinstance(out_q5[0], numbers.Real)
        Expecting:
            True
        **********************************************************************
  